# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. You will learn to:
- Load Croissant-formatted data packages
- Inspect record sets, fields, and columns using their `@id`s
- Extract and manipulate tables for analysis
- Apply filtering, normalization, and grouping
- Visualize data attributes

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Dataset identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we enumerate all record sets and key fields by their `@id` as defined in the Croissant schema.

In [ ]:
# List all record sets with their @id and fields
print("Available record sets:")
record_sets = list(dataset.record_sets())
for rs in record_sets:
    rs_meta = rs.metadata
    print(f"  - @id: {rs_meta['@id']}")
    print(f"    name: {rs_meta.get('name', '(no name)')}")
    if 'field' in rs_meta:
        field_list = rs_meta['field']
        if not isinstance(field_list, list):
            field_list = [field_list]
        print(f"    Fields:")
        for f in field_list:
            if isinstance(f, dict):
                print(f"      - @id: {f['@id']} -- name: {f.get('name','')} -- dataType: {f.get('dataType','')}")
            else:
                print(f"      - @id: {f}")
    else:
        print(f"    (No field list in metadata)")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Note:** All references use the record set and field `@id` values as discovered in the previous step.

In [ ]:
# Get a list of record set @ids
record_set_ids = [rs.metadata['@id'] for rs in dataset.record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records for current record set as DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if there are records
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display the list of record sets with DataFrames extracted
if dataframes:
    print(f"Record sets with loaded data:")
    for rsid, df in dataframes.items():
        print(f"- @id: {rsid} (shape: {df.shape})")
    # Pick the first one for further demo
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns of first record set @id '{first_rs_id}':\n{dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()
else:
    print("No record data was extracted from the record sets. Check that the dataset schema contains data links.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filter records by a numeric field
- Normalize a numeric field
- Group data by a categorical field

All field references use their Croissant `@id`.

In [ ]:
import numpy as np

# We'll use the first extracted record set (if available)
if dataframes:
    df = dataframes[first_rs_id]
    print(f"Example data from record set '{first_rs_id}':")
    display(df.head())
    
    # Find possible numeric fields (@id ending with e.g. '_value', or of type float or int)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_fields)==0:
        # Try to infer numeric fields
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_fields.append(col)
            except:
                continue
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        
        # Example: filter by threshold (use 10th percentile as threshold for demonstration)
        threshold = df[numeric_field_id].quantile(0.10)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by a categorical field, e.g. the first non-numeric column
        cat_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if cat_fields:
            group_field_id = cat_fields[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No numeric field available in the first record set for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All entity references use `@id` names whenever possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field (if data is available)
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()
    
    # If group_field_id exists, make bar plot of means
    if 'group_field_id' in locals():
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means = group_means.sort_values()
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² croissant dataset for rangeland knowledge adoption predictors in Northern Kenya. We loaded metadata, inspected record sets and fields by `@id`, extracted tables, filtered and normalized fields, grouped results, and visualized key distributions and relationships. This approach using `mlcroissant` fosters reproducible and structured access to FAIR and FAIR²-aligned datasets for transparent analytics.

*End of exploration.*